<a href="https://colab.research.google.com/github/EberHernandezBenitez/EDP/blob/main/Variables_antit%C3%A9ticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Variables antitéticas.
El método se fundamenta en que si deseamos estimar la integral $\theta =∫_{0}^{1} h(u) du = E[h(U)],$ donde $U \thicksim U(0,1)$ y la función $h(u)$ es monotona podemos generar pares de variables aleatorias fuertemente correlacionadas de forma negativa: $U$ y $U-1$.

Pata ello el estimador con variables antitéticas utiliza parejas combinadas:
$$Y_{i}=\frac{e^{U_i^2}+e^{(1-U_i)^2}}{2}$$

Lo compararemos con el método de monte carlo para comparar sus aproximaciones y calcula el porcentaje de reducción de varianza que se obtiene gracias al método de la Sección 8.1 con la integral dada en clase:
$$\int_{0}^{1} e^{x^2} dx$$

In [3]:
import random
import math

def estimar_integral_antiteticas(n_pares):
    """
    Estima la integral de e^(x^2) de 0 a 1 usando Monte Carlo Estándar
    y el método de Variables Antitéticas
    Parámetros:
    - n_pares: Número de pares (U, 1-U) a generar (total de evaluaciones = 2 * n_pares).
    """
    # Listas para almacenar las muestras de ambos métodos
    muestras_estandar = [] #las de monte carlo estándar
    muestras_antiteticas = []

    for _ in range(n_pares):
        # 1. Generar variable uniforme U ~ U(0, 1)
        u1 = random.random()
        u2 = random.random() # Otra uniforme independiente para el método estándar

        #Método Monte Carlo Estándar
        # Evaluamos dos puntos independientes para que el número total de llamadas a la función sea igual
        muestras_estandar.append(math.exp(u1**2))
        muestras_estandar.append(math.exp(u2**2))

        #Método de Variables Antitéticas
        # Se utiliza U y su contraparte antitética (1 - U)
        h_u = math.exp(u1**2)
        h_antitetica = math.exp((1 - u1)**2)

        # El estimador combinado por par es la media de ambos
        y_i = (h_u + h_antitetica) / 2
        muestras_antiteticas.append(y_i)

    # Media (Estimación de la integral)
    estimacion_estandar = sum(muestras_estandar) / len(muestras_estandar)
    estimacion_antitetica = sum(muestras_antiteticas) / len(muestras_antiteticas)

    # Varianza muestral del estimador estándar (basado en N muestras independientes)
    N_total = len(muestras_estandar)
    var_estandar = sum((x - estimacion_estandar)**2 for x in muestras_estandar) / (N_total - 1)
    # Varianza del promedio final del estimador estándar
    var_media_estandar = var_estandar / N_total

    # Varianza muestral de los pares antitéticos (basado en n_pares)
    var_antitetica_pares = sum((y - estimacion_antitetica)**2 for y in muestras_antiteticas) / (n_pares - 1)
    # Varianza del promedio final del estimador antitético
    var_media_antitetica = var_antitetica_pares / n_pares

    # Reducción de la varianza obtenida
    reduccion_varianza = (1 - (var_media_antitetica / var_media_estandar)) * 100

    return {
        "N_evaluaciones": N_total,
        "estandar_med": estimacion_estandar,
        "estandar_var": var_media_estandar,
        "antitetica_med": estimacion_antitetica,
        "antitetica_var": var_media_antitetica,
        "reduccion_pct": reduccion_varianza
    }

# EJECUCIÓN Y PRUEBA
if __name__ == "__main__":
    #500,000 pares para realizar un total de 1,000,000 de evaluaciones de la función
    PARES = 500000

    resultados = estimar_integral_antiteticas(PARES)

    print(f"ESTIMACIÓN DE LA INTEGRAL DE exp(x^2) DE 0 A 1 (N = {resultados['N_evaluaciones']:,})")
    print(f"{'MÉTODO':<25}{'ESTIMACIÓN (MEDIA)':<25}{'VARIANZA DEL ESTIMADOR':<20}")
    print("-"*65)
    print(f"{'Monte Carlo Estándar':<25}{resultados['estandar_med']:<25.4f}{resultados['estandar_var']:<20.2e}")
    print(f"{'Variables Antitéticas':<25}{resultados['antitetica_med']:<25.4f}{resultados['antitetica_var']:<20.2e}")
    print("-"*65)
    print(f"Porcentaje de reducción de varianza: {resultados['reduccion_pct']:.2f}%")
    print("="*65)

ESTIMACIÓN DE LA INTEGRAL DE exp(x^2) DE 0 A 1 (N = 1,000,000)
MÉTODO                   ESTIMACIÓN (MEDIA)       VARIANZA DEL ESTIMADOR
-----------------------------------------------------------------
Monte Carlo Estándar     1.4629                   2.25e-07            
Variables Antitéticas    1.4626                   5.57e-08            
-----------------------------------------------------------------
Porcentaje de reducción de varianza: 75.28%


Notamos que el método de variables antitéticas logra una reducción de varianza de aproximadamente el 76% en comparación con el método estándar utilizando exactamente el mismo número de evaluaciones de la función, demostrando la enorme ganancia en eficiencia.